## tl;dr

개방ID `2129314064`는 **안양대학교 대신신학대학원**과 일치한다. 2025년 EDSS에는 28개 패널 171행이 있고 재적 27명, 입학생 9명, 전임교원 3명, 학과 1개의 실제 활동이 확인된다. 이 값은 KEDI 2025 학교코드 `53079G59`의 값과 일치하며, 안양대학교는 2026학년도에도 이 대학원을 모집하고 있다.

권고 분류는 `confirmed_identity_active_id`이며, 학교코드 `53079G59`로 수동 연결해 활성기관 분석에 포함한다.

## Context & Methods

EDSS 제한 DB에서 해당 개방ID의 2024·2025 학교·입학·학과·패널 정보를 조회하고, 같은 연도의 KEDI 학교 원본과 수치를 대조했다. 현재 운영 여부는 안양대학교 공식 대학원 페이지와 2026학년도 모집 공지로 확인했다.

### Key Assumptions

- EDSS `panel_0101`의 입학생 수가 0이어도 대학정보공시 `panel_0306`에서 입학생이 확인되면 실제 입학 활동으로 본다.
- EDSS 지역 `경기 안양시 만안구`와 KEDI 지역 `경기 안양시`는 동일 캠퍼스의 서로 다른 행정구역 상세도다.
- 학교코드와 ID는 문자열로 다루며 원본 파일은 수정하지 않는다.

## Data

### 1. Load sources and helpers

In [1]:
from __future__ import annotations

import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import duckdb
from IPython.display import Markdown, display

OPEN_ID = "2129314064"
DB = Path("/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb")
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
KEDI_DIR = PROJECT_ROOT / "data/raw/kedi/higher_education_school"
OFFICIAL_PAGE = "https://www.anyang.ac.kr/main/university/graduate-school-of-theology.do"
OFFICIAL_2026_NOTICE = "https://www.anyang.ac.kr/grad4/commu/notice.do?article.offset=0&articleLimit=10&articleNo=40301&mode=view"
NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

def column_index(reference: str) -> int:
    result = 0
    for letter in re.match(r"[A-Z]+", reference).group(0):
        result = result * 26 + ord(letter) - 64
    return result - 1

def xlsx_records(path: Path) -> list[dict[str, str]]:
    with zipfile.ZipFile(path) as archive:
        shared_root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
        shared = [
            "".join(node.text or "" for node in item.findall(".//m:t", NS))
            for item in shared_root.findall("m:si", NS)
        ]
        sheet_root = ET.fromstring(archive.read("xl/worksheets/sheet1.xml"))
    rows = []
    for row in sheet_root.findall(".//m:row", NS):
        values = {}
        for cell in row.findall("m:c", NS):
            node = cell.find("m:v", NS)
            if node is None:
                continue
            values[column_index(cell.get("r"))] = shared[int(node.text)] if cell.get("t") == "s" else node.text
        rows.append([values.get(index, "") for index in range(max(values, default=-1) + 1)])
    header = rows[13]
    return [
        {header[index]: row[index] if index < len(row) else "" for index in range(len(header))}
        for row in rows[14:]
    ]

assert DB.exists()
assert all((KEDI_DIR / f"{year}_kedi_higher_education_school.xlsx").exists() for year in (2024, 2025))
con = duckdb.connect(str(DB), read_only=True)
print("Sources loaded for", OPEN_ID)

Sources loaded for 2129314064


## Results

### 2. Confirm annual coverage and school-level activity

In [2]:
tables = con.execute("""
SELECT table_schema, table_name FROM information_schema.columns
WHERE column_name='개방ID' AND table_schema IN ('higher_education', 'university_disclosure')
GROUP BY ALL ORDER BY ALL
""").fetchall()

panel_summary = []
for schema_name, table_name in tables:
    count = con.execute(
        f"SELECT COUNT(*) FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [OPEN_ID],
    ).fetchone()[0]
    if count:
        panel_summary.append((f"{schema_name}.{table_name}", count))

school_columns = [
    "조사년도", "지역명", "본분교명", "고등교육학교_입학생수", "고등교육학교_여자입학생수",
    "고등교육학교_졸업생수", "고등교육학교_여자졸업생수", "고등교육학교_교원수",
    "고등교육학교_여자교원수", "고등교육학교_재적학생수", "고등교육학교_재적여학생수",
    "고등교육학교_학과수",
]
school_rows = [dict(zip(school_columns, row)) for row in con.execute(
    f"SELECT {','.join(school_columns)} FROM higher_education.panel_0101 WHERE 개방ID=? ORDER BY 조사년도",
    [OPEN_ID],
).fetchall()]

display(Markdown(f"**2025 coverage:** {len(panel_summary)} panels, {sum(row[1] for row in panel_summary)} rows"))
display(school_rows)

**2025 coverage:** 28 panels, 171 rows

[{'조사년도': '2024',
  '지역명': '경기 안양시 만안구',
  '본분교명': '본교',
  '고등교육학교_입학생수': '0',
  '고등교육학교_여자입학생수': '0',
  '고등교육학교_졸업생수': '0',
  '고등교육학교_여자졸업생수': '0',
  '고등교육학교_교원수': '3',
  '고등교육학교_여자교원수': '0',
  '고등교육학교_재적학생수': '18',
  '고등교육학교_재적여학생수': '4',
  '고등교육학교_학과수': '1'},
 {'조사년도': '2025',
  '지역명': '경기 안양시 만안구',
  '본분교명': '본교(제1캠퍼스)',
  '고등교육학교_입학생수': '0',
  '고등교육학교_여자입학생수': '0',
  '고등교육학교_졸업생수': '0',
  '고등교육학교_여자졸업생수': '0',
  '고등교육학교_교원수': '3',
  '고등교육학교_여자교원수': '0',
  '고등교육학교_재적학생수': '27',
  '고등교육학교_재적여학생수': '7',
  '고등교육학교_학과수': '1'}]

### 3. Recover the entrant count from the disclosure panel

In [3]:
admission_rows = [dict(zip(
    ["조사년도", "학위과정구분명", "입학정원", "남자_정원내입학생", "여자_정원내입학생", "남자_정원외입학생", "여자_정원외입학생"], row
)) for row in con.execute("""
SELECT 조사년도, 학위과정구분명,
       신입생충원_대학원_입학정원수,
       신입생충원_대학원_정원내남자입학생수,
       신입생충원_대학원_정원내여자입학생수,
       신입생충원_대학원_정원외남자입학생수,
       신입생충원_대학원_정원외여자입학생수
FROM university_disclosure.panel_0306
WHERE 개방ID=? AND 조사년도 IN ('2024', '2025')
ORDER BY 조사년도, 학위과정구분명
""", [OPEN_ID]).fetchall()]

entrant_totals = {}
for year in ("2024", "2025"):
    year_rows = [row for row in admission_rows if row["조사년도"] == year]
    entrant_totals[year] = {
        "전체": sum(int(row[key]) for row in year_rows for key in ["남자_정원내입학생", "여자_정원내입학생", "남자_정원외입학생", "여자_정원외입학생"]),
        "여자": sum(int(row[key]) for row in year_rows for key in ["여자_정원내입학생", "여자_정원외입학생"]),
    }

display(admission_rows)
display(entrant_totals)

[{'조사년도': '2024',
  '학위과정구분명': '박사',
  '입학정원': '0',
  '남자_정원내입학생': '0',
  '여자_정원내입학생': '0',
  '남자_정원외입학생': '0',
  '여자_정원외입학생': '0'},
 {'조사년도': '2024',
  '학위과정구분명': '석박사통합',
  '입학정원': '0',
  '남자_정원내입학생': '0',
  '여자_정원내입학생': '0',
  '남자_정원외입학생': '0',
  '여자_정원외입학생': '0'},
 {'조사년도': '2024',
  '학위과정구분명': '석사',
  '입학정원': '30',
  '남자_정원내입학생': '9',
  '여자_정원내입학생': '1',
  '남자_정원외입학생': '1',
  '여자_정원외입학생': '0'},
 {'조사년도': '2025',
  '학위과정구분명': '박사',
  '입학정원': '0',
  '남자_정원내입학생': '0',
  '여자_정원내입학생': '0',
  '남자_정원외입학생': '0',
  '여자_정원외입학생': '0'},
 {'조사년도': '2025',
  '학위과정구분명': '석박사통합',
  '입학정원': '0',
  '남자_정원내입학생': '0',
  '여자_정원내입학생': '0',
  '남자_정원외입학생': '0',
  '여자_정원외입학생': '0'},
 {'조사년도': '2025',
  '학위과정구분명': '석사',
  '입학정원': '30',
  '남자_정원내입학생': '6',
  '여자_정원내입학생': '3',
  '남자_정원외입학생': '0',
  '여자_정원외입학생': '0'}]

{'2024': {'전체': 11, '여자': 1}, '2025': {'전체': 9, '여자': 3}}

### 4. Inspect department labels

In [4]:
department_rows = [dict(zip(
    ["조사년도", "학과명", "학과상태명", "학위과정구분명", "주야간계절구분명"], row
)) for row in con.execute("""
SELECT DISTINCT 조사년도, 학과명, 학과상태명, 학위과정구분명, 주야간계절구분명
FROM university_disclosure.panel_1017
WHERE 개방ID=?
ORDER BY 조사년도, 학과명, 학위과정구분명
""", [OPEN_ID]).fetchall()]

display(department_rows)

[{'조사년도': '2024',
  '학과명': '목회학과',
  '학과상태명': '기존',
  '학위과정구분명': '박사',
  '주야간계절구분명': '주간'},
 {'조사년도': '2024',
  '학과명': '목회학과',
  '학과상태명': '기존',
  '학위과정구분명': '석박사통합',
  '주야간계절구분명': '주간'},
 {'조사년도': '2024',
  '학과명': '목회학과',
  '학과상태명': '기존',
  '학위과정구분명': '석사',
  '주야간계절구분명': '주간'},
 {'조사년도': '2024',
  '학과명': '신학과',
  '학과상태명': '기존',
  '학위과정구분명': '박사',
  '주야간계절구분명': '주간'},
 {'조사년도': '2024',
  '학과명': '신학과',
  '학과상태명': '기존',
  '학위과정구분명': '석박사통합',
  '주야간계절구분명': '주간'},
 {'조사년도': '2024',
  '학과명': '신학과',
  '학과상태명': '기존',
  '학위과정구분명': '석사',
  '주야간계절구분명': '주간'},
 {'조사년도': '2025',
  '학과명': '목회학과',
  '학과상태명': '기존',
  '학위과정구분명': '박사',
  '주야간계절구분명': '주간'},
 {'조사년도': '2025',
  '학과명': '목회학과',
  '학과상태명': '기존',
  '학위과정구분명': '석박사통합',
  '주야간계절구분명': '주간'},
 {'조사년도': '2025',
  '학과명': '목회학과',
  '학과상태명': '기존',
  '학위과정구분명': '석사',
  '주야간계절구분명': '주간'},
 {'조사년도': '2025',
  '학과명': '신학과',
  '학과상태명': '기존',
  '학위과정구분명': '박사',
  '주야간계절구분명': '주간'},
 {'조사년도': '2025',
  '학과명': '신학과',
  '학과상태명': '기존',
  '학위과정구분명': '석박사통합',
  

### 5. Match the KEDI candidate

In [5]:
candidate_fields = [
    "학교코드", "학교명", "학제", "대학원구분", "학교상태", "시군구", "본분교",
    "재적생_전체_계", "재적생_전체_여", "입학자_전체_계", "입학자_전체_여",
    "졸업자_전체_계", "졸업자_전체_여", "전임교원_계", "전임교원_여", "학과수_전체",
]
kedi_candidates = {}
for year in (2024, 2025):
    candidates = [
        {key: row.get(key, "") for key in candidate_fields}
        for row in xlsx_records(KEDI_DIR / f"{year}_kedi_higher_education_school.xlsx")
        if row.get("시군구", "").startswith("경기 안양")
        and row.get("대학원구분") == "부설대학원"
        and "대신신학" in row.get("학교명", "")
    ]
    kedi_candidates[year] = candidates

display(kedi_candidates)

{2024: [{'학교코드': '',
   '학교명': '안양대학교 대신신학대학원',
   '학제': '특수대학원',
   '대학원구분': '부설대학원',
   '학교상태': '기존',
   '시군구': '경기 안양시',
   '본분교': '본교(제1캠퍼스)',
   '재적생_전체_계': '18',
   '재적생_전체_여': '4',
   '입학자_전체_계': '11',
   '입학자_전체_여': '1',
   '졸업자_전체_계': '0',
   '졸업자_전체_여': '0',
   '전임교원_계': '3',
   '전임교원_여': '0',
   '학과수_전체': '1'}],
 2025: [{'학교코드': '53079G59',
   '학교명': '안양대학교 대신신학대학원',
   '학제': '특수대학원',
   '대학원구분': '부설대학원',
   '학교상태': '기존',
   '시군구': '경기 안양시',
   '본분교': '본교(제1캠퍼스)',
   '재적생_전체_계': '27',
   '재적생_전체_여': '7',
   '입학자_전체_계': '9',
   '입학자_전체_여': '3',
   '졸업자_전체_계': '0',
   '졸업자_전체_여': '0',
   '전임교원_계': '3',
   '전임교원_여': '0',
   '학과수_전체': '1'}]}

### 6. Run decision-critical checks

In [6]:
assert len(panel_summary) == 28
assert sum(row[1] for row in panel_summary) == 171
assert {row["조사년도"] for row in school_rows} == {"2024", "2025"}
assert entrant_totals == {"2024": {"전체": 11, "여자": 1}, "2025": {"전체": 9, "여자": 3}}
assert {row["학과명"] for row in department_rows} == {"목회학과", "신학과"}
assert {row["학과상태명"] for row in department_rows} == {"기존"}
assert {row["주야간계절구분명"] for row in department_rows} == {"주간"}
assert len(kedi_candidates[2024]) == 1 and len(kedi_candidates[2025]) == 1
assert kedi_candidates[2025][0]["학교코드"] == "53079G59"

for year in (2024, 2025):
    edss = next(row for row in school_rows if row["조사년도"] == str(year))
    kedi = kedi_candidates[year][0]
    assert edss["고등교육학교_재적학생수"] == kedi["재적생_전체_계"]
    assert edss["고등교육학교_재적여학생수"] == kedi["재적생_전체_여"]
    assert edss["고등교육학교_교원수"] == kedi["전임교원_계"]
    assert edss["고등교육학교_학과수"] == kedi["학과수_전체"]
    assert entrant_totals[str(year)]["전체"] == int(kedi["입학자_전체_계"])
    assert entrant_totals[str(year)]["여자"] == int(kedi["입학자_전체_여"])

print("All decision-critical checks passed.")

All decision-critical checks passed.


### 7. Official active-status evidence

- [안양대학교 대신신학대학원 공식 소개](https://www.anyang.ac.kr/main/university/graduate-school-of-theology.do)는 목회학과와 신학과를 운영한다고 명시한다.
- [2026학년도 후기 모집 전형 공지](https://www.anyang.ac.kr/grad4/commu/notice.do?article.offset=0&articleLimit=10&articleNo=40301&mode=view)는 대신신학대학원 석사과정 지원자의 2026년 6월 면접 일정을 안내한다.

따라서 2025년뿐 아니라 2026년에도 운영이 이어지는 활성 대학원이다.

## Takeaways

1. `2129314064`는 안양대학교 대신신학대학원과 2024·2025년 핵심 수치가 일치한다.
2. 2025 KEDI 학교코드는 `53079G59`, 학교상태는 `기존`이다.
3. EDSS 2025에는 28개 패널 171행과 다수의 실제 활동값이 있으며, 폐교·폐과 잔존 ID가 아니다.
4. 자동매칭 실패는 EDSS의 더 상세한 지역명과 `panel_0101` 입학생 0 때문에 발생했다. 입학생은 `panel_0306`에서 KEDI와 일치한다.
5. `confirmed_identity_active_id`로 분류하고 `53079G59`에 수동 연결해 활성기관 분석에 포함하는 것이 안전하다.